# Phase 1: DistilGPT-2 Decoder Smoke Tests

Zero-shot prompting on Jigsaw comments:
- **D1** — per-label yes/no questions
- **D2** — sentence completion (`This comment is ___`) for all 6 labels + argmax probe

Default `QUICK_MODE = True` (`20+20` eval sample). Set `QUICK_MODE = False` for the full `50+50` run.

In [ ]:
%matplotlib inline

import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch

QUICK_MODE = True
THRESHOLD = 0.5

# Walk up from cwd until we find this repo's src/paths.py (avoids wrong root → None paths).
_src = None
for _p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    _cand = _p / "src"
    if (_cand / "paths.py").is_file():
        _src = str(_cand)
        break
if _src is None:
    raise FileNotFoundError(
        f"Cannot find DeepProjectHate/src/paths.py from cwd={Path.cwd()}. "
        "Open the notebook from the project folder."
    )
if _src not in sys.path:
    sys.path.insert(0, _src)

for _mod in ("paths", "device", "data", "eval_sample", "decoder", "roster"):
    sys.modules.pop(_mod, None)

from data import add_toxic_any, load_train_df, verify_data_files
from decoder import (
    D2_LABEL_CONTINUATIONS,
    build_d1_prompt,
    build_d2_prompt,
    d1_display_df,
    d1_label_detail,
    d1_method_summary,
    d2_argmax_summary,
    d2_display_df,
    d2_label_detail,
    d2_method_summary,
    format_method_summary_text,
    method_mean_by_method,
    predict_multilabel_d1,
    predict_multilabel_d2,
    score_yes_no,
)
from device import get_device, sync_if_cuda
from eval_sample import get_balanced_sample, get_smoke_gold_df, get_smoke_texts, show_random_rows
from paths import (
    DECODER_SMOKE_SCORES_PATH,
    DISTILGPT2_MODEL_ID,
    LOCAL_DISTILGPT2_DIR,
    PROJECT_ROOT,
    SEED,
    TRAIN_CSV,
    TOXIC_LABELS,
    eval_n_each,
)
from roster import require_path

np.random.seed(SEED)
torch.manual_seed(SEED)

device = get_device()
print(f"Project root: {PROJECT_ROOT}")
print(f"src on path: {_src}")
print(f"TRAIN_CSV: {TRAIN_CSV}")
print(f"Device: {device} ({'GPU' if device.type == 'cuda' else 'CPU'})")
print(f"QUICK_MODE: {QUICK_MODE} (eval n_each={eval_n_each(QUICK_MODE)})")
print(f"PyTorch: {torch.__version__}")


## 1. Load data + balanced eval sample

In [ ]:
# Guard: PathLike NoneType usually means TRAIN_CSV was None (wrong sys.path / stale kernel).
train_path = require_path(TRAIN_CSV, "TRAIN_CSV")
if not train_path.is_file():
    raise FileNotFoundError(
        f"Missing train CSV at {train_path}. Run: python scripts/download_jigsaw.py"
    )

verify_data_files()
train_df = add_toxic_any(load_train_df())
sample = get_balanced_sample(train_df, quick_mode=QUICK_MODE)
texts = sample["comment_text"].astype(str).tolist()

print(f"Loaded {train_path}")
print(f"Eval sample: {len(sample)} rows ({sample['toxic'].sum()} toxic, {(sample['toxic']==0).sum()} non-toxic)")


### Install model dependencies (run once)

If needed: `%pip install -r ../requirements-project.txt`, then restart kernel.

In [ ]:
# Uncomment and run once if needed:
# %pip install -r ../requirements-project.txt

## 2. Load DistilGPT-2

Prefetch: `python scripts/download_distilgpt2.py`

In [ ]:
from transformers import GPT2LMHeadModel, GPT2Tokenizer

if LOCAL_DISTILGPT2_DIR.exists():
    model_path = str(LOCAL_DISTILGPT2_DIR)
else:
    model_path = DISTILGPT2_MODEL_ID
if not isinstance(model_path, str) or not model_path:
    raise TypeError(
        f"DistilGPT-2 path/id is {model_path!r}. "
        "Expected local models/distilgpt2 or DISTILGPT2_MODEL_ID='distilgpt2'. "
        "Re-run setup cell or: python scripts/download_distilgpt2.py"
    )
print(f"Loading from: {model_path}")

tokenizer = GPT2Tokenizer.from_pretrained(model_path)
model = GPT2LMHeadModel.from_pretrained(model_path).to(device)
model.eval()
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

n_params = sum(p.numel() for p in model.parameters())
print(f"Parameters: {n_params:,}")


## 3. D1 — yes/no prompting (smoke)

For **each of the 6 labels**, we ask the decoder an explicit question and score how it continues after `Answer:`.

**Example prompt (toxic):**
```
Is the following comment toxic? Answer yes or no.
Comment: {text}
Answer:
```

We compare four scoring paths:
- **logprob_full** — log P(` yes`) vs log P(` no`) over full tokens
- **logprob_first_token (C)** — first token only
- **constrained (D)** — greedy decode restricted to yes/no token ids
- **generate_parse (B)** — free generation; abstain if not parseable as yes/no

In [ ]:
print("D1 example prompt:")
print(build_d1_prompt(get_smoke_texts()[0], "toxic"))
print("-" * 60)

smoke_texts = get_smoke_texts()
smoke_d1 = predict_multilabel_d1(model, tokenizer, smoke_texts, device)
display(show_random_rows(d1_label_detail(smoke_d1, "toxic"), n=len(smoke_texts)).round(3))

smoke_d1_eval = smoke_d1.merge(get_smoke_gold_df(), on="comment_text")
smoke_d1_summary = d1_method_summary(smoke_d1_eval)
display(smoke_d1_summary.round(3))
print(format_method_summary_text(smoke_d1_summary, len(smoke_texts), prompt_name="D1 smoke"))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.barplot(data=smoke_d1_summary, x="label", y="agreement", hue="method", ax=axes[0])
axes[0].set_ylim(0, 1)
axes[0].set_title("D1 smoke: agreement vs reference gold by label")
plt.setp(axes[0].xaxis.get_majorticklabels(), rotation=30, ha="right")

sns.barplot(data=method_mean_by_method(smoke_d1_summary), x="method", y="mean_agreement", ax=axes[1], color="steelblue")
axes[1].set_ylim(0, 1)
axes[1].set_title("D1 smoke: mean agreement by method")
plt.setp(axes[1].xaxis.get_majorticklabels(), rotation=20, ha="right")
plt.tight_layout()
plt.show()

## 4. D2 — sentence completion (smoke)

We ask the decoder to **complete the sentence** after:
```
Comment: {text}
This comment is ___
```

For each label we score a **positive** vs **negative** continuation using symmetric phrasing
(e.g. toxic: ` toxic` vs ` not toxic` — same frame as D1 yes/no).
Same four methods as D1.

We also run an **argmax probe**: which label's positive phrase is the single most likely
next token? (mutually exclusive — diagnostic only, not true multi-label).

In [ ]:
print("D2 example prompt:")
print(build_d2_prompt(get_smoke_texts()[0]))
print("Continuation pairs per label:")
for lbl, (pos, neg) in D2_LABEL_CONTINUATIONS.items():
    print(f"  {lbl}: '{pos.strip()}' vs '{neg.strip()}'")
print("-" * 60)

smoke_d2 = predict_multilabel_d2(model, tokenizer, smoke_texts, device)
display(show_random_rows(d2_label_detail(smoke_d2, "toxic"), n=len(smoke_texts)).round(3))

smoke_d2_eval = smoke_d2.merge(get_smoke_gold_df(), on="comment_text")
smoke_d2_summary = d2_method_summary(smoke_d2_eval)
display(smoke_d2_summary.round(3))
print(format_method_summary_text(smoke_d2_summary, len(smoke_texts), prompt_name="D2 smoke"))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.barplot(data=smoke_d2_summary, x="label", y="agreement", hue="method", ax=axes[0])
axes[0].set_ylim(0, 1)
axes[0].set_title("D2 smoke: agreement vs reference gold by label")
plt.setp(axes[0].xaxis.get_majorticklabels(), rotation=30, ha="right")

sns.barplot(data=method_mean_by_method(smoke_d2_summary), x="method", y="mean_agreement", ax=axes[1], color="coral")
axes[1].set_ylim(0, 1)
axes[1].set_title("D2 smoke: mean agreement by method")
plt.setp(axes[1].xaxis.get_majorticklabels(), rotation=20, ha="right")
plt.tight_layout()
plt.show()

## 5. D1 — balanced eval sample

6 prompts per comment (61 columns in `d1_df`). Slow on CPU — use `QUICK_MODE` while debugging.

**Column map (per label):**

| Method | Key columns | Notes |
|--------|-------------|-------|
| logprob_full (A) | `d1_p_yes_{l}`, `d1_pred_logprob_{l}` | Primary; pred = p ≥ 0.5 |
| first_token (C) | `d1_p_yes_first_{l}`, `d1_pred_logprob_first_{l}` | First-token only |
| constrained (D) | `d1_p_yes_constrained_{l}`, `d1_pred_constrained_{l}` | Greedy yes/no |
| generate_parse (B) | `d1_pred_generate_{l}`, `d1_parse_abstain_{l}`, `d1_raw_completion_{l}` | May abstain |

Below: **primary preds only** (`d1_display_df`). Full wide table is kept for §7+ and parquet.

In [ ]:
print(f"Running D1 on {len(texts)} comments x {len(TOXIC_LABELS)} labels ...")
d1_df = predict_multilabel_d1(model, tokenizer, texts, device)
print(f"Full d1_df: {len(d1_df.columns)} columns (all methods × labels). Showing primary preds:")
display(show_random_rows(d1_display_df(d1_df), n=5))

## 6. D2 — balanced eval sample

Same comment, 6 completion pairs + argmax label per row.

In [ ]:
print(f"Running D2 on {len(texts)} comments x {len(TOXIC_LABELS)} labels + argmax ...")
d2_df = predict_multilabel_d2(model, tokenizer, texts, device)
print(f"Full d2_df: {len(d2_df.columns)} columns. Showing primary preds + argmax:")
display(show_random_rows(d2_display_df(d2_df, include_argmax=True), n=5))

## 7. D2 argmax — most likely single label

Picks the label whose positive continuation has highest first-step log-probability.
**Not multi-label** — at most one winner per comment.

In [ ]:
compare = sample[["comment_text"] + TOXIC_LABELS].copy()
compare = compare.merge(d1_df, on="comment_text").merge(d2_df, on="comment_text")

argmax_counts = compare["d2_argmax_label"].value_counts().rename_axis("argmax_label").reset_index(name="count")
display(argmax_counts)

argmax_diag = d2_argmax_summary(compare)
display(argmax_diag.round(3))

top_argmax = argmax_counts.iloc[0]
toxic_argmax = argmax_diag.loc[argmax_diag["gold_label"] == "toxic", "argmax_rate_when_gold"]
toxic_argmax_txt = f"{toxic_argmax.iloc[0]:.0%}" if len(toxic_argmax) else "n/a"
print(
    f"Argmax picks '{top_argmax['argmax_label']}' most often ({top_argmax['count']}/{len(compare)} comments). "
    f"When gold toxic=1, argmax equals 'toxic' {toxic_argmax_txt} of the time."
)

fig, ax = plt.subplots(figsize=(7, 4))
sns.countplot(data=compare, x="d2_argmax_label", order=TOXIC_LABELS, ax=ax, color="steelblue")
ax.set_title("D2 argmax label distribution (eval sample)")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

## 8. Sanity check — toxic label (D1 vs D2)

In [ ]:
compare["d1_p_toxic"] = compare["d1_p_yes_toxic"]
compare["d1_pred_toxic"] = compare["d1_pred_logprob_toxic"]
compare["d2_p_toxic"] = compare["d2_p_positive_toxic"]
compare["d2_pred_toxic"] = compare["d2_pred_logprob_toxic"]

for prefix, pred_col in [("D1 logprob", "d1_pred_toxic"), ("D2 logprob", "d2_pred_toxic")]:
    agree = (compare["toxic"] == compare[pred_col]).mean()
    print(f"{prefix} agreement vs gold toxic: {agree:.1%}")

d1_toxic_agree = (compare["toxic"] == compare["d1_pred_toxic"]).mean()
d2_toxic_agree = (compare["toxic"] == compare["d2_pred_toxic"]).mean()
winner = "D1" if d1_toxic_agree >= d2_toxic_agree else "D2"
print(
    f"On {len(compare)} balanced comments, {winner} logprob_full matches gold toxic more often "
    f"({max(d1_toxic_agree, d2_toxic_agree):.1%} vs {min(d1_toxic_agree, d2_toxic_agree):.1%})."
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
plot_df = compare.assign(gold_label=compare["toxic"].map({0: "non-toxic", 1: "toxic"}))
for ax, score_col, title in [
    (axes[0], "d1_p_toxic", "D1 P(yes) for toxic"),
    (axes[1], "d2_p_toxic", "D2 P(positive) for toxic"),
]:
    sns.violinplot(data=plot_df, x="gold_label", y=score_col, hue="gold_label", ax=ax, palette=["seagreen", "indianred"], legend=False)
    ax.axhline(THRESHOLD, color="crimson", linestyle="--", linewidth=1)
    ax.set_title(title)
plt.tight_layout()
plt.show()

## 9. D1 scoring methods — agreement by label

In [ ]:
d1_method_df = d1_method_summary(compare)
display(d1_method_df.round(3))
print(format_method_summary_text(d1_method_df, len(compare), prompt_name="D1 eval"))

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
sns.barplot(data=d1_method_df, x="label", y="agreement", hue="method", ax=axes[0])
axes[0].set_ylim(0, 1)
axes[0].set_title("D1 agreement with gold by method")
plt.setp(axes[0].xaxis.get_majorticklabels(), rotation=30, ha="right")

sns.barplot(data=method_mean_by_method(d1_method_df), x="method", y="mean_agreement", ax=axes[1], color="steelblue")
axes[1].set_ylim(0, 1)
axes[1].set_title("D1 mean agreement by method")
plt.setp(axes[1].xaxis.get_majorticklabels(), rotation=20, ha="right")

abstain_df = d1_method_df[d1_method_df["method"] == "generate_parse"]
sns.barplot(data=abstain_df, x="label", y="abstain_rate", ax=axes[2], color="gray")
axes[2].set_ylim(0, 1)
axes[2].set_title("D1 generate_parse abstain rate")
plt.setp(axes[2].xaxis.get_majorticklabels(), rotation=30, ha="right")
plt.tight_layout()
plt.show()

## 10. D2 scoring methods — agreement by label

In [ ]:
d2_method_df = d2_method_summary(compare)
display(d2_method_df.round(3))
print(format_method_summary_text(d2_method_df, len(compare), prompt_name="D2 eval"))

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
sns.barplot(data=d2_method_df, x="label", y="agreement", hue="method", ax=axes[0])
axes[0].set_ylim(0, 1)
axes[0].set_title("D2 agreement with gold by method")
plt.setp(axes[0].xaxis.get_majorticklabels(), rotation=30, ha="right")

sns.barplot(data=method_mean_by_method(d2_method_df), x="method", y="mean_agreement", ax=axes[1], color="coral")
axes[1].set_ylim(0, 1)
axes[1].set_title("D2 mean agreement by method")
plt.setp(axes[1].xaxis.get_majorticklabels(), rotation=20, ha="right")

abstain_df = d2_method_df[d2_method_df["method"] == "generate_parse"]
sns.barplot(data=abstain_df, x="label", y="abstain_rate", ax=axes[2], color="gray")
axes[2].set_ylim(0, 1)
axes[2].set_title("D2 generate_parse abstain rate")
plt.setp(axes[2].xaxis.get_majorticklabels(), rotation=30, ha="right")
plt.tight_layout()
plt.show()

## 11. Scoring mismatches (logprob_full vs generate_parse)

In [ ]:
mismatch_rows = []
for label in TOXIC_LABELS:
    mismatch_rows.append({
        "label": label,
        "d1_mismatch": compare[f"d1_scoring_mismatch_{label}"].mean(),
        "d2_mismatch": compare[f"d2_scoring_mismatch_{label}"].mean(),
    })
mismatch_df = pd.DataFrame(mismatch_rows)
display(mismatch_df.round(3))

d1_mm = mismatch_df["d1_mismatch"].mean()
d2_mm = mismatch_df["d2_mismatch"].mean()
d1_abstain = compare[[f"d1_parse_abstain_{l}" for l in TOXIC_LABELS]].stack().mean()
d2_abstain = compare[[f"d2_parse_abstain_{l}" for l in TOXIC_LABELS]].stack().mean()
print(
    f"Mean scoring mismatch (logprob vs generate): D1 {d1_mm:.1%}, D2 {d2_mm:.1%}. "
    f"Mean parse abstain: D1 {d1_abstain:.1%}, D2 {d2_abstain:.1%}."
)

melted = mismatch_df.melt(id_vars="label", var_name="prompt_type", value_name="mismatch_rate")
fig, ax = plt.subplots(figsize=(9, 4))
sns.barplot(data=melted, x="label", y="mismatch_rate", hue="prompt_type", ax=ax)
ax.set_title("logprob_full != generate_parse (non-abstain only)")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

## 12. Inference micro-benchmark

In [ ]:
bench_text = texts[0]
bench_prompt = build_d1_prompt(bench_text, "toxic")
bench_repeats = 50 if QUICK_MODE else 200

for _ in range(3):
    score_yes_no(model, tokenizer, bench_prompt, device)
sync_if_cuda(device)

t0 = time.perf_counter()
for _ in range(bench_repeats):
    score_yes_no(model, tokenizer, bench_prompt, device)
sync_if_cuda(device)
ms_per_prompt = 1000 * (time.perf_counter() - t0) / bench_repeats

print(f"Model: {DISTILGPT2_MODEL_ID}")
print(f"Parameters: {n_params:,}")
print(f"Device: {device}")
print(f"D1 single prompt (1 label): {ms_per_prompt:.2f} ms")
print(f"D1 all 6 labels (estimate): {ms_per_prompt * 6:.2f} ms/comment")
print(f"D2 all 6 labels + argmax (estimate): {ms_per_prompt * 7:.2f} ms/comment")

## 13. Save decoder smoke scores

In [ ]:
out = compare.copy()
out["quick_mode"] = QUICK_MODE
out["model_id"] = DISTILGPT2_MODEL_ID
out["device"] = str(device)
out["threshold"] = THRESHOLD

DECODER_SMOKE_SCORES_PATH.parent.mkdir(parents=True, exist_ok=True)
out.to_parquet(DECODER_SMOKE_SCORES_PATH, index=False)
print(f"Saved {len(out)} rows to {DECODER_SMOKE_SCORES_PATH}")

## Next steps

- Join `encoder_smoke_scores.parquet` (notebook 01) with `decoder_smoke_scores.parquet`
- Deferred: formal eval notebook with train/val/test split
- Phase 2: `run_benchmark.py` — D3 few-shot, prompt ablations